# Experiment 7 — Domain adaptation: finetune the synthetic-pretrained model (exp3) on `dataset_matan`

Warm-starts from **`cyttic/trocr-hebrew-synthetic-cont-unfrozen`** (the exp3
model: pretrained on ~125k synthetic Hebrew lines, **bridge already trained**,
encoder already adapted to Hebrew) and finetunes it on
**`cyttic/trocr-hebrew-matan`** (~4,400 real handwritten lines, writer-disjoint
train/test).

**Why this is a different problem from exp4-6.** Those started from
`trocr-hebrew-untrained`, whose visual→text cross-attention is *random noise* —
asking 4,400 lines to teach a blind model to see from scratch. They all failed
the same way (CER pinned ~0.85-1.0; see the experiments report). exp7 starts from
a model that **already reads Hebrew** (CER ~6% on synthetic). So this is
**domain adaptation** — nudging a working reader from rendered synthetic ink to
real handwriting — not training a bridge from scratch. It should actually work.

**Configuration (single unfrozen phase):**

| Setting | Value | Why |
|---|---|---|
| Encoder | **unfrozen** | synthetic→real is a *visual* shift; the encoder must adapt to real strokes. Safe now — the bridge is no longer random |
| Learning rate | **2e-5** | a converged model + tiny data wants gentle nudges, not the 5e-5 from-scratch rate |
| Effective batch | **16** (8 × grad-accum 2) | exp3's proven unfrozen footprint on the L4 |
| Epochs | **12**, eval each epoch | enough to adapt; watch for overfit |
| Best-model capture | **on** (`load_best_model_at_end`, by CER) | **the key change vs exp4-6** — keep the min-CER epoch before overfit sets in |
| Label smoothing | 0.0 (knob exposed) | keep the first run clean/interpretable; turn on 0.1 if it overfits fast |

**What to watch:** unlike exp4-6, you should see **CER actually trend down** for
the first epochs, then validation loss start rising as it overfits — that's
success + overfit, not failure. `load_best_model_at_end` grabs the best epoch
automatically. CER is on held-out **new writers**, so calibrate expectations to
that.

**Target hardware:** L4 (24 GB, bf16). Checkpoints push to the Hub each epoch
(resume-safe); the **best** model is saved + pushed at the end.

## 1. Setup

Self-contained: downloads the model + dataset from HuggingFace. Run top to bottom.

In [ ]:
# Run once on a fresh VM/Colab runtime. Comment out if deps are already installed.
!pip install -q torch transformers datasets accelerate jiwer sacrebleu pillow matplotlib

In [ ]:
import os
# Set BEFORE importing torch: lets the allocator grow/shrink segments instead of
# fragmenting, which is the failure the exp5 OOM hint pointed at for the unfrozen phase.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import math
import torch
import jiwer
import sacrebleu
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    VisionEncoderDecoderModel,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint
from huggingface_hub import snapshot_download
from huggingface_hub.utils import RepositoryNotFoundError

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("alloc :", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
if device == "cuda":
    print("gpu   :", torch.cuda.get_device_name(0))
    print("vram  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
    print("bf16  :", torch.cuda.is_bf16_supported())

In [ ]:
# --- HuggingFace auth (Colab has no persistent box auth -> log in every session) ---
# Needs a WRITE token. Create one at https://huggingface.co/settings/tokens (token type: Write)
from huggingface_hub import notebook_login, whoami
notebook_login()
print("logged in as:", whoami()["name"])

In [ ]:
# --- HebrewBlockProcessor (inlined so this notebook is self-contained) ---
from PIL import Image, ImageOps

class HebrewBlockProcessor:
    """Mirror (RTL->LTR) -> resize to 64px height -> tile into a 384x384 ViT container."""
    TARGET_HEIGHT = 64
    CONTAINER_SIZE = 384
    IMAGE_MEAN = [0.5, 0.5, 0.5]
    IMAGE_STD = [0.5, 0.5, 0.5]

    def __call__(self, images, return_tensors="pt"):
        if not isinstance(images, list):
            images = [images]
        pixel_values = torch.stack([self._process(img) for img in images])
        return {"pixel_values": pixel_values}

    def _process(self, image):
        image = image.convert("RGB")
        image = ImageOps.mirror(image)
        w, h = image.size
        new_w = max(1, round(w * self.TARGET_HEIGHT / h))
        image = image.resize((new_w, self.TARGET_HEIGHT), Image.LANCZOS)
        container = Image.new("RGB", (self.CONTAINER_SIZE, self.CONTAINER_SIZE), (255, 255, 255))
        img_arr = np.array(image)
        src_x, dest_x, dest_y = 0, 0, 0
        while src_x < new_w and dest_y < self.CONTAINER_SIZE:
            chunk_w = min(new_w - src_x, self.CONTAINER_SIZE - dest_x)
            chunk = Image.fromarray(img_arr[:, src_x:src_x + chunk_w])
            container.paste(chunk, (dest_x, dest_y))
            src_x += chunk_w
            dest_x += chunk_w
            if dest_x >= self.CONTAINER_SIZE:
                dest_x = 0
                dest_y += self.TARGET_HEIGHT
        t = torch.tensor(np.array(container), dtype=torch.float32).permute(2, 0, 1) / 255.0
        mean = torch.tensor(self.IMAGE_MEAN).view(3, 1, 1)
        std = torch.tensor(self.IMAGE_STD).view(3, 1, 1)
        return (t - mean) / std

## 2. Config

Single unfrozen phase. `LR = 2e-5` (the main knob — drop to 1e-5 if it overfits
in the first epochs). `LABEL_SMOOTHING` is left at 0.0 for a clean first run.

In [ ]:
MODEL_ID   = "cyttic/trocr-hebrew-synthetic-cont-unfrozen"  # exp3: synthetic-pretrained, bridge already trained
DATASET_ID = "cyttic/trocr-hebrew-matan"                    # writer-level train/test split

EPOCHS     = 12
BATCH_SIZE = 8          # unfrozen on L4 -> batch 8 (exp3's proven footprint)
GRAD_ACCUM = 2          # effective batch size = 8 x 2 = 16

LR              = 2e-5  # gentle: we're nudging a converged reader, not training from scratch
WARMUP_RATIO    = 0.1
WEIGHT_DECAY    = 0.01
LABEL_SMOOTHING = 0.0   # set 0.1 if it overfits fast (loss will then plateau higher -- expected)

MAX_TARGET_LENGTH = 128
NUM_WORKERS = 4
PRECISION = "bf16"      # target hardware: L4 (24GB, bf16). Use "fp16" on a Turing card (RTX 2080).
MAX_STEPS = -1          # set e.g. 50 for a quick smoke test

RUN_NAME   = "trocr-hebrew-matan-exp7"
OUTPUT_DIR = f"output/{RUN_NAME}"
BEST_DIR   = f"{OUTPUT_DIR}/best"

HUB_CKPTS_REPO = f"cyttic/{RUN_NAME}-ckpts"   # mid-run checkpoints (resume)
HUB_REPO       = f"cyttic/{RUN_NAME}"         # FINAL best-by-CER model

SAVE_LIMIT = 2

print(f"base   : {MODEL_ID}")
print(f"train  : {EPOCHS} epochs, encoder UNFROZEN, lr={LR}, eff.batch={BATCH_SIZE}x{GRAD_ACCUM}")
print(f"best-by: eval_cer (load_best_model_at_end)")
print("ckpts  :", HUB_CKPTS_REPO, "| final:", HUB_REPO)

## 3. Model, tokenizer, processor

Loaded from the **exp3 synthetic-pretrained** model (`MODEL_ID`) — its encoder,
cross-attention bridge and decoder are all already trained. The encoder stays
trainable for the single unfrozen domain-adaptation phase below.

In [ ]:
model     = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
processor = HebrewBlockProcessor()

# generation_config takes priority over model.config in recent transformers,
# so set the special tokens on it explicitly or generate() fails during eval.
def reset_generation_config(m):
    m.generation_config.decoder_start_token_id = tokenizer.cls_token_id
    m.generation_config.pad_token_id = tokenizer.pad_token_id
    m.generation_config.eos_token_id = tokenizer.sep_token_id
    m.generation_config.max_new_tokens = None

reset_generation_config(model)

def report_trainable(m, tag=""):
    n_total = sum(p.numel() for p in m.parameters())
    n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{tag} params total: {n_total/1e6:.1f}M | trainable: {n_train/1e6:.1f}M "
          f"({100*n_train/n_total:.1f}%)")

report_trainable(model, "[loaded]")

## 4. Dataset

In [ ]:
ds = load_dataset(DATASET_ID)
print(ds)

eval_ds = ds["test"]
print("eval (test) samples:", len(eval_ds))

## 5. Sanity checks

Confirm the `train`/`test` split is **writer-level / leak-free** (no writer's
lines appear in both, the same principle `CLAUDE.md` requires for the human
set), then *see* what `HebrewBlockProcessor` does to a real line.

In [ ]:
tr_writers = set(ds["train"]["writer"])
te_writers = set(ds["test"]["writer"])
print(f"train writers: {len(tr_writers)} | test writers: {len(te_writers)} | "
      f"OVERLAP: {len(tr_writers & te_writers)} (should be 0)")

In [ ]:
sample = ds["train"][0]
img = sample["image"].convert("RGB")
print("writer:", sample["writer"], "| text:", sample["text"])

pv = processor([img])["pixel_values"][0]
shown = (pv * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].imshow(img);   ax[0].set_title("raw matan line"); ax[0].axis("off")
ax[1].imshow(shown); ax[1].set_title("after HebrewBlockProcessor (mirrored + tiled 384x384)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 6. Collator + metrics (CER / WER for per-epoch eval)

In [ ]:
def collate(batch):
    images = [ex["image"].convert("RGB") for ex in batch]
    texts  = [ex["text"] for ex in batch]
    pixel_values = processor(images)["pixel_values"]
    labels = tokenizer(
        texts, padding="longest", truncation=True,
        max_length=MAX_TARGET_LENGTH, return_tensors="pt",
    ).input_ids
    labels[labels == tokenizer.pad_token_id] = -100
    return {"pixel_values": pixel_values, "labels": labels}


def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    pred_ids  = np.where(pred_ids  < 0, tokenizer.pad_token_id, pred_ids)
    label_ids = np.where(label_ids < 0, tokenizer.pad_token_id, label_ids)
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"cer": jiwer.cer(label_str, pred_str),
            "wer": jiwer.wer(label_str, pred_str)}


def make_targs():
    return Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        label_smoothing_factor=LABEL_SMOOTHING,
        num_train_epochs=EPOCHS,
        max_steps=MAX_STEPS,
        bf16=(PRECISION == "bf16"),
        fp16=(PRECISION == "fp16"),
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        generation_num_beams=1,
        eval_strategy="epoch",
        save_strategy="epoch",          # must match eval_strategy for load_best_model_at_end
        logging_steps=50,
        save_total_limit=SAVE_LIMIT,
        load_best_model_at_end=True,    # <-- keep the best epoch, not the last
        metric_for_best_model="eval_cer",
        greater_is_better=False,        # lower CER is better
        dataloader_num_workers=NUM_WORKERS,
        remove_unused_columns=False,
        report_to="none",
        push_to_hub=True,
        hub_model_id=HUB_CKPTS_REPO,
        hub_strategy="checkpoint",
        hub_private_repo=True,
    )


def resolve_resume_checkpoint(local_dir, hub_ckpts_repo):
    # Local checkpoint first; else pull 'last-checkpoint' from the Hub (Colab has no durable disk).
    ckpt = get_last_checkpoint(local_dir) if os.path.isdir(local_dir) else None
    if ckpt is None:
        try:
            snapshot_download(hub_ckpts_repo, repo_type="model",
                              local_dir=local_dir, allow_patterns="last-checkpoint/*")
            cand = os.path.join(local_dir, "last-checkpoint")
            ckpt = cand if os.path.isdir(cand) else None
        except RepositoryNotFoundError:
            pass  # no checkpoints pushed yet
    return ckpt

## 7. Train — single unfrozen phase (domain adaptation)

The exp3 base already has all parameters trainable; we make that explicit, then
finetune for `EPOCHS` epochs on matan. `load_best_model_at_end=True` means after
training `model` holds the **lowest-CER epoch's** weights (not necessarily the
last) — those are what we save and push.

Checkpoints push to `HUB_CKPTS_REPO` every epoch, so a lost Colab session resumes
from the last epoch (re-run this cell — `resolve_resume_checkpoint` finds it).

In [ ]:
for p in model.encoder.parameters():
    p.requires_grad = True          # exp3 base is already fully trainable; explicit for clarity
report_trainable(model, "[exp7 / unfrozen domain-adapt]")

targs = make_targs()
trainer = Seq2SeqTrainer(
    model=model, args=targs,
    train_dataset=ds["train"], eval_dataset=eval_ds,
    data_collator=collate, compute_metrics=compute_metrics,
)

last_ckpt = resolve_resume_checkpoint(OUTPUT_DIR, HUB_CKPTS_REPO)
print("Resuming from", last_ckpt) if last_ckpt else print("Starting exp7 from the exp3 (synthetic) weights")
trainer.train(resume_from_checkpoint=last_ckpt)

# load_best_model_at_end=True -> `model` is now the best-CER epoch
trainer.save_model(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)
print("best model saved ->", BEST_DIR)

model.push_to_hub(HUB_REPO)
tokenizer.push_to_hub(HUB_REPO)
print("best model pushed ->", HUB_REPO)

## 8. Final CER / WER / BLEU (full beam-search pass)

Runs once on the **best** model (loaded above) over the full matan test split
with beam search — this is the headline result for exp7.

In [ ]:
@torch.no_grad()
def eval_cer_wer_bleu(model, dataset, beams=4, batch_size=BATCH_SIZE, max_length=MAX_TARGET_LENGTH, tag=""):
    model.eval()
    refs, hyps = [], []
    for start in range(0, len(dataset), batch_size):
        batch = dataset[start:start + batch_size]
        imgs = [im.convert("RGB") for im in batch["image"]]
        pv = processor(imgs)["pixel_values"].to(model.device, dtype=model.dtype)
        ids = model.generate(pv, num_beams=beams, max_length=max_length)
        hyps.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
        refs.extend(batch["text"])
        done = min(start + batch_size, len(dataset))
        if done % (batch_size * 5) == 0 or done == len(dataset):
            print(f"  {tag} {done}/{len(dataset)}", flush=True)

    cer = jiwer.cer(refs, hyps)
    wer = jiwer.wer(refs, hyps)
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    exact = sum(r.strip() == h.strip() for r, h in zip(refs, hyps)) / len(refs)
    return {"cer": cer, "wer": wer, "bleu": bleu, "exact": exact, "refs": refs, "hyps": hyps}


print(f"Running FINAL eval (best exp7 model) on {len(eval_ds)} matan test lines, beam=4 ...")
final = eval_cer_wer_bleu(model, eval_ds, beams=4, tag="[final]")
print(f"\nFINAL exp7 (best-by-CER)  |  N={len(eval_ds)}  |  "
      f"CER {final['cer']*100:.2f}%  WER {final['wer']*100:.2f}%  "
      f"BLEU {final['bleu']:.2f}  |  exact {final['exact']*100:.2f}%")

## 9. Look at predictions

Spot-check the best model on a handful of held-out matan lines (GT vs PRED).

In [ ]:
model.eval()
n = 6
fig, axes = plt.subplots(n, 1, figsize=(10, 2.2 * n))
for ax, ex in zip(axes, ds["test"].select(range(n))):
    img = ex["image"].convert("RGB")
    pv = processor([img])["pixel_values"].to(model.device, dtype=model.dtype)
    with torch.no_grad():
        ids = model.generate(pv, num_beams=4, max_new_tokens=MAX_TARGET_LENGTH)
    pred = tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"GT  : {ex['text']}\nPRED: {pred}", loc="left", fontsize=9)
plt.tight_layout(); plt.show()